In [4]:
import os
import json
import numpy as np
import cv2
from pathlib import Path
from tqdm import tqdm

CLASS_MAP = {
    "background": 0,           # 배경
    "crack": 1,                # 단순 균열
    "leak": 2,                 # 누수
    "efflorescence": 3,        # 백태
    "detachment": 4,           # 들뜸
    "reticular crack": 5,      # 망상 균열
    "spalling": 6,             # 박리
    "material separation": 7,  # 재료 분리
    "rebar": 8,                # 철근 노출
    "damage": 9,               # 파손
    "exhilaration": 10         # 박락
}

def generate_segmentation_masks(json_base_dir, output_mask_dir, max_crack_samples=20000):
    
    os.makedirs(output_mask_dir, exist_ok=True)
    
    json_paths = list(Path(json_base_dir).rglob("*.json"))
    
    crack_count = 0
    other_count = 0
    
    for json_path in tqdm(json_paths):
        with open(json_path, 'r', encoding='utf-8') as f:
            try:
                data = json.load(f)
            except json.JSONDecodeError:
                continue
        
        img_info = data.get("image", {})
        if img_info.get("object_included", "N") == "N":
            continue
            
        annotations = img_info.get("annotations", [])
        if not annotations:
            continue
            
        unique_labels = set(ann.get("label", "").lower() for ann in annotations if ann.get("label"))
        
        if unique_labels == {"crack"}:
            if crack_count >= max_crack_samples:
                continue
            crack_count += 1
        else:
            other_count += 1 
            
        width = img_info.get("width", 1920)
        height = img_info.get("height", 1080)
        mask = np.zeros((height, width), dtype=np.uint8)
        
        for ann in annotations:
            label_name = ann.get("label", "").lower()
            class_id = CLASS_MAP.get(label_name, 0)
            
            if class_id == 0: continue
                
            points = ann.get("points", [])
            if not points: continue
            
            pts = np.array(points, np.int32).reshape((-1, 1, 2))
            shape_type = ann.get("shape", "Polygon")
            
            if shape_type == "Polyline":
                cv2.polylines(mask, [pts], isClosed=False, color=class_id, thickness=5)
            else:
                cv2.fillPoly(mask, [pts], color=class_id)
                
        original_img_name = img_info.get("name", json_path.stem + ".jpg")
        mask_filename = Path(original_img_name).stem + ".png"
        mask_save_path = os.path.join(output_mask_dir, mask_filename)
        
        cv2.imwrite(mask_save_path, mask)

    print(f" 단순 균열(Crack) 마스크 : {crack_count:,}장")
    print(f" 기타 결함 마스크       : {other_count:,}장")
    print(f" 총 마스크       : {crack_count + other_count:,}장")
    print(f" 저장 위치: {output_mask_dir}")

JSON_BASE_DIR = r"D:\Study\HumanStudy\Dataset\Training\02.라벨링데이터"
OUTPUT_MASK_DIR = r"D:\Study\HumanStudy\Dataset\Training_Masks" 

generate_segmentation_masks(JSON_BASE_DIR, OUTPUT_MASK_DIR, max_crack_samples=20000)

100%|█████████████████████████████████████████████████████████████████████████| 419995/419995 [45:20<00:00, 154.40it/s]


 단순 균열(Crack) 마스크 : 20,000장
 기타 결함 마스크       : 78,398장
 총 마스크       : 98,398장
 저장 위치: D:\Study\HumanStudy\Dataset\Training_Masks
